# Realistic institutional constraints

This notebook combines the smooth nonlinear objective with a realistic long-only mandate:

- fully invested;
- one fixed factor exposure;
- long-only and per-name caps;
- sector lower and upper bounds;
- hard two-way turnover.

The PGD projection is the Euclidean projection onto the **joint intersection**. Dykstra's algorithm
cycles over analytic projectors while retaining correction terms, so this is not naive sequential
clipping.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda value: f"{value:,.8g}")

In [ ]:
from portfolio_pgd import (
    ConstraintSet,
    PGDOptions,
    PortfolioProblem,
    PowerLawCost,
    capped_long_only_portfolio,
    factor_covariance,
    sector_membership,
    solve_pgd,
    solve_scipy_slsqp,
)

n_assets = 40
n_sectors = 5
rng = np.random.default_rng(3301)
covariance, loadings = factor_covariance(n_assets, 5, seed=3302, specific_risk=0.18)
previous = capped_long_only_portfolio(n_assets, cap=0.045, seed=3303)
sectors = sector_membership(n_assets, n_sectors)
previous_sector = sectors @ previous

# Sector bands are centered on the existing portfolio, making feasibility explicit.
sector_lower = np.maximum(0.12, previous_sector - 0.035)
sector_upper = np.minimum(0.30, previous_sector + 0.035)
A_ub = np.vstack([sectors, -sectors])
b_ub = np.concatenate([sector_upper, -sector_lower])

factor_direction = loadings[:, 0] - np.mean(loadings[:, 0])
A_eq = np.vstack([np.ones(n_assets), factor_direction])
b_eq = np.array([1.0, float(factor_direction @ previous)])

constraints = ConstraintSet(
    n_assets,
    equality_matrix=A_eq,
    equality_target=b_eq,
    inequality_matrix=A_ub,
    inequality_upper=b_ub,
    lower_bounds=0.0,
    upper_bounds=0.06,
    turnover_limit=0.22,
    turnover_center=previous,
)

problem = PortfolioProblem(
    alpha=rng.normal(scale=0.035, size=n_assets),
    covariance=covariance,
    previous_holdings=previous,
    risk_aversion=1.6,
    quadratic_cost_matrix=0.2 + rng.random(n_assets),
    quadratic_cost_aversion=0.25,
    nonlinear_cost=PowerLawCost(eta=0.006 + 0.006 * rng.random(n_assets), p=1.5, epsilon=1e-3),
)
print("Starting portfolio violations:", constraints.violations(previous))

In [ ]:
pgd = solve_pgd(
    problem,
    constraints,
    options=PGDOptions(
        max_iterations=30_000,
        tolerance=1e-7,
        projection_tolerance=2e-10,
    ),
)
slsqp = solve_scipy_slsqp(problem, constraints, tolerance=1e-10)

comparison = pd.DataFrame(
    {
        "objective": [pgd.objective, slsqp.objective],
        "utility": [pgd.utility, -slsqp.objective],
        "turnover": [np.sum(np.abs(pgd.trades)), np.sum(np.abs(slsqp.holdings - previous))],
        "distance_to_SLSQP": [np.linalg.norm(pgd.holdings - slsqp.holdings), 0.0],
        "constraint_violation": [
            constraints.max_violation(pgd.holdings),
            constraints.max_violation(slsqp.holdings),
        ],
    },
    index=["PGD", "SciPy SLSQP"],
)
print(comparison.to_string())
print(f"\nPGD status={pgd.status}; iterations={pgd.iterations}; SLSQP={slsqp.message}")

## Constraint audit

In [ ]:
audit = pd.DataFrame(
    {
        "PGD violation": constraints.violations(pgd.holdings),
        "SLSQP violation": constraints.violations(slsqp.holdings),
    }
)
print(audit.to_string())

sector_exposure = sectors @ pgd.holdings
sector_table = pd.DataFrame(
    {
        "lower": sector_lower,
        "previous": previous_sector,
        "optimized": sector_exposure,
        "upper": sector_upper,
    },
    index=[f"Sector {index}" for index in range(n_sectors)],
)
print("\nSector exposures:\n", sector_table.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
asset_index = np.arange(n_assets)
axes[0].plot(asset_index, previous, "o-", label="Previous", markersize=3)
axes[0].plot(asset_index, pgd.holdings, "o-", label="Optimized", markersize=3)
axes[0].axhline(0.06, color="black", linestyle="--", linewidth=1, label="Per-name cap")
axes[0].set(title="Holdings", xlabel="Asset", ylabel="Weight")
axes[0].legend()
axes[1].bar(asset_index, pgd.trades)
axes[1].axhline(0.0, color="black", linewidth=0.8)
axes[1].set(title=f"Trades (L1={np.sum(np.abs(pgd.trades)):.4f})", xlabel="Asset", ylabel="Weight change")
plt.tight_layout()
plt.show()

sector_table.plot(kind="bar", figsize=(11, 4))
plt.title("Sector exposure audit")
plt.ylabel("Portfolio weight")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Optimization convergence

In [ ]:
history = pd.DataFrame(pgd.history)
valid = history["projected_gradient_norm"].notna()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["iteration"], history["objective"])
axes[0].axhline(slsqp.objective, color="black", linestyle="--", label="SLSQP")
axes[0].set(title="Objective", xlabel="Iteration", ylabel="Negative utility")
axes[0].legend()
axes[1].semilogy(
    history.loc[valid, "iteration"],
    np.maximum(history.loc[valid, "projected_gradient_norm"], 1e-18),
)
axes[1].set(title="Projected-gradient residual", xlabel="Iteration", ylabel="Norm")
plt.tight_layout()
plt.show()

## Long-short extension

The same projection engine can construct a dollar-neutral, beta-neutral portfolio with per-name and
gross-exposure limits. This remains convex; gross exposure is an $L_1$ ball centered at zero.

In [ ]:
long_short_constraints = ConstraintSet(
    n_assets,
    equality_matrix=np.vstack([np.ones(n_assets), loadings[:, 1]]),
    equality_target=np.array([0.0, 0.0]),
    lower_bounds=-0.08,
    upper_bounds=0.08,
    gross_exposure_limit=1.0,
)
long_short_problem = PortfolioProblem(
    alpha=2.5 * problem.alpha,
    covariance=problem.covariance,
    previous_holdings=np.zeros(n_assets),
    risk_aversion=0.9,
    quadratic_cost_matrix=np.ones(n_assets),
    quadratic_cost_aversion=0.08,
)
long_short = solve_pgd(
    long_short_problem,
    long_short_constraints,
    options=PGDOptions(max_iterations=25_000, tolerance=1e-8),
)
print(
    pd.Series(
        {
            "status": long_short.status,
            "net exposure": np.sum(long_short.holdings),
            "gross exposure": np.sum(np.abs(long_short.holdings)),
            "beta exposure": loadings[:, 1] @ long_short.holdings,
            "maximum position": np.max(np.abs(long_short.holdings)),
            "constraint violation": long_short.max_constraint_violation,
        }
    ).to_string()
)

## Executable acceptance tests

In [ ]:
assert pgd.converged
assert slsqp.success
assert constraints.max_violation(pgd.holdings) < 2e-7
assert constraints.max_violation(slsqp.holdings) < 2e-6
assert abs(pgd.objective - slsqp.objective) < 2e-5
assert np.sum(np.abs(pgd.trades)) <= 0.22 + 2e-7
assert long_short.converged
assert long_short_constraints.max_violation(long_short.holdings) < 2e-7
print("All realistic-constraint validation checks passed.")